In [2]:
import sys
import logging
from typing import Dict, Any, List, Optional, Annotated
from typing_extensions import TypedDict

# 导入真正的 LangGraph 与消息组件
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("PolicyGateGraphV2")

# ============================================================================
# 1. 真实底层工具定义 (使用 @tool 装饰器)
# ============================================================================

class MockBackendState:
    def __init__(self, balance: float):
        self.balance = balance
        self.debit_count = 0

# 全局或者通过闭包绑定的后端实例
backend_state = MockBackendState(balance=500.0)

@tool
def get_balance() -> dict:
    """查询当前账户余额"""
    logger.info("⚙️ [Tool Execution] 正在执行只读工具: get_balance()")
    return {"balance": backend_state.balance}

@tool
def execute_transfer(amount: float, recipient: str) -> dict:
    """执行转账写操作副作用工具"""
    logger.warning(f"🚨 [Tool Execution] 正在执行高危工具: execute_transfer(amount={amount}, recipient='{recipient}')")
    if amount > backend_state.balance:
        return {"success": False, "reason": "INSUFFICIENT_FUNDS"}
    backend_state.balance -= amount
    backend_state.debit_count += 1
    return {"success": True, "remaining_balance": backend_state.balance}

# 将工具注册给标准的 ToolNode
tools = [get_balance, execute_transfer]
tool_node_executor = ToolNode(tools)

# ============================================================================
# 2. 状态定义 (AgentState)
# ============================================================================

class AgentState(TypedDict):
    messages: List[BaseMessage]       # 真实 LangChain 消息历史
    approved_tool_call_ids: List[str] # 精确绑定到 specific tool_call_id 的授权列表
    filtered_ai_message: Optional[AIMessage] # 供 ToolNode 专用的已过滤 AIMessage

# ============================================================================
# 3. 节点逻辑定义 (Nodes)
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """
    1. Agent 决策节点：模拟 LLM 一次性生成包含只读工具与写操作工具的 AIMessage
    """
    logger.info("🤖 [agent_node] LLM 正在生成复合 Tool Calls (get_balance + execute_transfer)...")
    
    # 构造包含多个 Tool Call 的真实 AIMessage
    ai_msg = AIMessage(
        content="请帮我查一下余额，并转账 1000 元给 Alice。",
        tool_calls=[
            {"id": "call_balance_01", "name": "get_balance", "args": {}},
            {"id": "call_transfer_01", "name": "execute_transfer", "args": {"amount": 1000.0, "recipient": "Alice"}}
        ]
    )
    return {"messages": [ai_msg]}


def policy_gate_node(state: AgentState) -> AgentState:
    """
    2. Policy Gate 独立安全审查与过滤节点：
    - 独立审查每个 tool_call。
    - 只读工具 (get_balance) 默认无条件 ALLOW。
    - 副作用工具 (execute_transfer) 必须严格检查：
      1) 是否有对应的精确 tool_call_id 审批授权 (approved_tool_call_ids)。
      2) 重新在执行前检查余额状态（防止顺序导致的逻辑漏洞）。
    - 隔离出允许执行的 tool_calls 构建专属于 ToolNode 的 AIMessage，并为被拦截的工具补全 ToolMessage。
    """
    logger.info("🛡️ [policy_gate_node] 正在进行细粒度契约审查与安全过滤...")
    
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return {}

    tool_calls = last_message.tool_calls
    approved_ids = state.get("approved_tool_call_ids", [])
    
    allowed_tool_calls = []
    generated_tool_messages = []

    for tc in tool_calls:
        t_id = tc["id"]
        t_name = tc["name"]
        
        if t_name == "get_balance":
            # 只读工具直接放行
            allowed_tool_calls.append(tc)
        elif t_name == "execute_transfer":
            # 高危副作用工具：必须有精确的 id 审批授权
            if t_id not in approved_ids:
                logger.warning(f"🛑 [Policy Gate Interception] 转账工具 [{t_id}] 未获得精确授权，已被拦截！")
                generated_tool_messages.append(
                    ToolMessage(content="ERROR: REJECTED_BY_POLICY_GATE_UNAUTHORIZED", tool_call_id=t_id)
                )
            else:
                # 即使获得了审批，也需要进行前置余额防御性校验
                amount = tc["args"].get("amount", 0.0)
                if amount > backend_state.balance:
                    logger.warning(f"🛑 [Policy Gate Interception] 转账金额 ({amount}) 大于当前余额 ({backend_state.balance})，安全网关主动取消！")
                    generated_tool_messages.append(
                        ToolMessage(content="ERROR: CANCELLED_DUE_TO_INSUFFICIENT_BALANCE", tool_call_id=t_id)
                    )
                else:
                    logger.info(f"✅ [Policy Gate Approved] 转账工具 [{t_id}] 审批与前置校验通过，准予执行。")
                    allowed_tool_calls.append(tc)

    # 构造一个仅包含被允许工具调用的 AIMessage 传给真正的 ToolNode
    filtered_ai = AIMessage(content=last_message.content, tool_calls=allowed_tool_calls)
    
    # 将被拦截工具的占位 ToolMessage 和过滤后的消息合并回历史，保持历史完整
    new_messages = [last_message] + generated_tool_messages
    
    return {
        "filtered_ai_message": filtered_ai,
        "messages": new_messages
    }


def safe_execution_router(state: AgentState) -> str:
    """
    条件路由函数：判断经过 Policy Gate 过滤后，是否还有工具需要进入 ToolNode 执行。
    """
    filtered_ai = state.get("filtered_ai_message")
    if filtered_ai and len(filtered_ai.tool_calls) > 0:
        return "execute_tools"
    return "skip_tools"


def custom_tool_executor_node(state: AgentState) -> AgentState:
    """
    3. 工具执行节点：将过滤后的 AIMessage 替换到消息末尾并交由标准的 ToolNode 执行
    """
    logger.info("⚡ [custom_tool_executor_node] 正在调度标准 ToolNode 执行合规工具...")
    filtered_ai = state["filtered_ai_message"]
    
    # 临时把历史消息末尾替换为仅含合法 tool_calls 的 AIMessage，以满足 ToolNode 上下文要求
    original_messages = state["messages"]
    state["messages"][-1] = filtered_ai
    
    # 调用 LangGraph 原生 ToolNode
    result = tool_node_executor.invoke(state)
    
    # 恢复或追加生成的 ToolMessage
    return {"messages": result["messages"]}


def skip_execution_node(state: AgentState) -> AgentState:
    """
    没有合法工具需要执行时的兜底节点
    """
    logger.info("⏭️ [skip_execution_node] 没有合法的 Tool Call 需要执行，跳过 ToolNode。")
    return {}

# ============================================================================
# 4. 构建 LangGraph 状态图 (StateGraph Compilation)
# ============================================================================

def build_agent_graph():
    builder = StateGraph(AgentState)
    
    # 注册节点
    builder.add_node("agent", agent_node)
    builder.add_node("policy_gate", policy_gate_node)
    builder.add_node("tool_node", custom_tool_executor_node)
    builder.add_node("skip_node", skip_execution_node)
    
    # 添加边与条件路由
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "policy_gate")
    
    builder.add_conditional_edges(
        "policy_gate",
        safe_execution_router,
        {
            "execute_tools": "tool_node",
            "skip_tools": "skip_node"
        }
    )
    
    builder.add_edge("tool_node", END)
    builder.add_edge("skip_node", END)
    
    return builder.compile()

# ============================================================================
# 5. 测试场景与断言验证
# ============================================================================

def run_scenario_a():
    print("\n" + "=" * 70)
    print("🧪 场景 A 验证: 余额 500，申请转账 1000 (预期: 自动前置拦截/余额不足取消，扣款 0 次)")
    print("=" * 70)
    
    # 重置后端状态
    global backend_state
    backend_state = MockBackendState(balance=500.0)
    
    # 即使给出了精确授权 id，由于余额只有 500 而申请转账 1000，Policy Gate 应该在执行前拦截
    initial_state: AgentState = {
        "messages": [],
        "approved_tool_call_ids": ["call_transfer_01"],
        "filtered_ai_message": None
    }
    
    graph = build_agent_graph()
    result = graph.invoke(initial_state)
    
    print(f"\n--- 场景 A 结果 ---")
    print(f"最终账户余额: {backend_state.balance}")
    print(f"实际扣款次数 (debit_count): {backend_state.debit_count}")
    print(f"消息历史条数: {len(result['messages'])}")
    
    # 断言
    assert backend_state.debit_count == 0, f"安全漏洞：余额不足时依然发生了扣款！次数: {backend_state.debit_count}"
    assert backend_state.balance == 500.0, "余额应当保持不变"
    print("✅ 场景 A 断言全部通过：成功取消转账，扣款 0 次。")


def run_scenario_b():
    print("\n" + "=" * 70)
    print("🧪 场景 B 验证: 余额 5000，申请转账 1000，未经批准 (预期: 拒绝执行，扣款 0 次)")
    print("=" * 70)
    
    # 重置后端状态：余额充足 (5000)
    global backend_state
    backend_state = MockBackendState(balance=5000.0)
    
    # 关键点：approved_tool_call_ids 为空，代表未经人工审批授权
    initial_state: AgentState = {
        "messages": [],
        "approved_tool_call_ids": [], # 未授权任何写操作
        "filtered_ai_message": None
    }
    
    graph = build_agent_graph()
    result = graph.invoke(initial_state)
    
    print(f"\n--- 场景 B 结果 ---")
    print(f"最终账户余额: {backend_state.balance}")
    print(f"实际扣款次数 (debit_count): {backend_state.debit_count}")
    
    # 断言
    assert backend_state.debit_count == 0, f"严重安全漏洞：未经批准的转账被执行了！扣款次数: {backend_state.debit_count}"
    assert backend_state.balance == 5000.0, "余额应当保持不变"
    print("✅ 场景 B 断言全部通过：余额足够但未经批准，成功拦截，扣款 0 次。")

if __name__ == "__main__":
    run_scenario_a()
    run_scenario_b()

2026-10-20 14:19:37,752 - [INFO] - 🤖 [agent_node] LLM 正在生成复合 Tool Calls (get_balance + execute_transfer)...
2026-10-20 14:19:37,755 - [INFO] - 🛡️ [policy_gate_node] 正在进行细粒度契约审查与安全过滤...
2026-10-20 14:19:37,758 - [WARNING] - 🛑 [Policy Gate Interception] 转账金额 (1000.0) 大于当前余额 (500.0)，安全网关主动取消！
2026-10-20 14:19:37,761 - [INFO] - ⚡ [custom_tool_executor_node] 正在调度标准 ToolNode 执行合规工具...
2026-10-20 14:19:37,766 - [INFO] - ⚙️ [Tool Execution] 正在执行只读工具: get_balance()
2026-10-20 14:19:37,783 - [INFO] - 🤖 [agent_node] LLM 正在生成复合 Tool Calls (get_balance + execute_transfer)...
2026-10-20 14:19:37,785 - [INFO] - 🛡️ [policy_gate_node] 正在进行细粒度契约审查与安全过滤...
2026-10-20 14:19:37,785 - [WARNING] - 🛑 [Policy Gate Interception] 转账工具 [call_transfer_01] 未获得精确授权，已被拦截！
2026-10-20 14:19:37,787 - [INFO] - ⚡ [custom_tool_executor_node] 正在调度标准 ToolNode 执行合规工具...
2026-10-20 14:19:37,790 - [INFO] - ⚙️ [Tool Execution] 正在执行只读工具: get_balance()



🧪 场景 A 验证: 余额 500，申请转账 1000 (预期: 自动前置拦截/余额不足取消，扣款 0 次)

--- 场景 A 结果 ---
最终账户余额: 500.0
实际扣款次数 (debit_count): 0
消息历史条数: 1
✅ 场景 A 断言全部通过：成功取消转账，扣款 0 次。

🧪 场景 B 验证: 余额 5000，申请转账 1000，未经批准 (预期: 拒绝执行，扣款 0 次)

--- 场景 B 结果 ---
最终账户余额: 5000.0
实际扣款次数 (debit_count): 0
✅ 场景 B 断言全部通过：余额足够但未经批准，成功拦截，扣款 0 次。
